In [1]:
import os
import pyodbc
import pandas as pd

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)

In [13]:
query_Lop = "SELECT Ten_lop FROM Lop"
df_Lop = pd.read_sql(query_Lop, conn_libol)
print(df_Lop)

       Ten_lop
0      191040B
1      191040A
2    19109CL1B
3    19109CL1A
4    19109CL2B
..         ...
597    211611B
598    211611A
599    211612A
600    211612B
601      21950

[602 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21780\4172466584.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Lop = pd.read_sql(query_Lop, conn_libol)


In [ ]:
query_LopBandoc = "SELECT DISTINCT dbo.DecodeUTF8String(Lop) AS Lop FROM Ban_doc"
df_LopBandoc = pd.read_sql(query_LopBandoc, conn_libol)
print(df_LopBandoc)

C:\Users\admin\AppData\Local\Temp\ipykernel_21780\548940832.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_LopBandoc = pd.read_sql(query_LopBandoc, conn_libol)


           Lop
0             
1            0
2           00
3          000
4       001011
...        ...
4259   ÊN11021
4260  ên14010A
4261  ên2D02VD
4262      đIỆN
4263      None

[4264 rows x 1 columns]


In [ ]:
df_data_lop = pd.DataFrame({"Ten_lop": pd.concat([df_Lop["Ten_lop"], 
                                                  df_LopBandoc["Lop"]], 
                                                  ignore_index=True)})
df_data_lop = df_data_lop.dropna(subset=['Ten_lop'])
df_data_lop = df_data_lop.drop_duplicates().reset_index(drop=True)
df_data_lop = df_data_lop.sort_values(by="Ten_lop", ascending=True).reset_index(drop=True)
print(df_data_lop)


       Ten_lop
0             
1            0
2           00
3          000
4       001011
...        ...
4260   ÊN11021
4261  ên14010A
4262  ên2D02VD
4263      đIỆN
4264      None

[4265 rows x 1 columns]


In [5]:
df_Dim_lop = df_Lop.copy()

# Tạo một danh sách để lưu các dòng mới cần thêm
new_rows = []
current_max_id = df_Dim_lop["ID"].max()

# Duyệt qua các khoa trong df_Khoa
for i, lop_ban_doc in enumerate(df_LopBandoc['Lop']):
    is_existing = False
    # Duyệt qua các dòng trong df_data_khoa để kiểm tra sự tồn tại
    for j, row in df_Lop.iterrows():
        lop = row['Ten_lop'].lower().strip() if pd.notna(row['Ten_lop']) else ""
        
        # Kiểm tra nếu khoa hiện tại trùng khớp hoặc chứa một chuỗi trong chuỗi kia
        if (lop_ban_doc == lop):
            is_existing = True
            break  # Nếu đã tìm thấy trùng khớp, không cần kiểm tra thêm

    # Nếu không tồn tại, thêm khoa vào danh sách new_rows
    if not is_existing:
        current_max_id += 1
        new_rows.append({"ID": current_max_id, "Ten_lop": lop_ban_doc})

# Sử dụng pd.concat để hợp nhất các dòng mới với df_Dim_Khoa
df_Dim_lop = pd.concat([df_Dim_lop, pd.DataFrame(new_rows)], ignore_index=True)

# Hiển thị kết quả
print(df_Dim_lop)


        ID    Ten_lop
0        1    191040B
1        2    191040A
2        3  19109CL1B
3        4  19109CL1A
4        5  19109CL2B
...    ...        ...
4859  4860        709
4860  4861   14151CLC
4861  4862    181311A
4862  4863  19143CL1B
4863  4864      23950

[4864 rows x 2 columns]


In [6]:
file_path = './Mapping_Lop.csv'
if os.path.exists(file_path):
    os.remove(file_path)
df_Dim_lop.to_csv(file_path, index=False, encoding='utf-8-sig')